# Session 9 exercises in Python (recap, Session 10)

The recap notes set up each problem: the states, $P$, the boundary conditions and one equation
per state. This notebook does the rest where Python makes sense.

- **SymPy** solves the linear systems exactly, so the answers come out as fractions, as in the notes.
- **NumPy** is used for simulation and matrix powers.

Exercise 1, the classification parts of Exercises 3 and 5, and the interpretation in Exercise 6
are done by hand in the notes.

**Watch out:** in SymPy `P**n` is the matrix power. In NumPy `P**n` raises every *entry* to the
power n. In NumPy always use `P @ P` or `np.linalg.matrix_power(P, n)`.

In [1]:
import numpy as np
import sympy as sp
from sympy import Rational as R

np.set_printoptions(precision=4, suppress=True)

## 1. Exercise 3 b): probability of eventual release

Order (Rev, Test, Rel, Rej). Rel and Rej are absorbing, so $a_{\text{Rel}} = 1$ and
$a_{\text{Rej}} = 0$. First-step analysis for the two transient states (rows Rev and Test of $P$):

$$
\begin{aligned}
a_{\text{Rev}} &= 0.3\,a_{\text{Rev}} + 0.6\,a_{\text{Test}} + 0.1\cdot 0\\
a_{\text{Test}} &= 0.2\,a_{\text{Rev}} + 0.3\,a_{\text{Test}} + 0.4\cdot 1 + 0.1\cdot 0
\end{aligned}
$$

Write the probabilities with `R(3, 10)` instead of `0.3`: then SymPy keeps exact fractions.

In [2]:
a_Rev, a_Test = sp.symbols("a_Rev a_Test")

eqs_a = [sp.Eq(a_Rev,  R(3, 10)*a_Rev + R(6, 10)*a_Test + R(1, 10)*0),
         sp.Eq(a_Test, R(2, 10)*a_Rev + R(3, 10)*a_Test + R(4, 10)*1 + R(1, 10)*0)]

sol_a = sp.solve(eqs_a, [a_Rev, a_Test])
sol_a

{a_Rev: 24/37, a_Test: 28/37}

In [3]:
print("a_Rev  =", sol_a[a_Rev],  "≈", round(float(sol_a[a_Rev]), 3))
print("a_Test =", sol_a[a_Test], "≈", round(float(sol_a[a_Test]), 3))

a_Rev  = 24/37 ≈ 0.649
a_Test = 28/37 ≈ 0.757


## 2. Exercise 3 c): expected number of steps until absorption

$A = \{\text{Rel}, \text{Rej}\}$, so $t_{\text{Rel}} = t_{\text{Rej}} = 0$. For the transient
states, $t_i = 1 + \sum_k p_{ik}\,t_k$:

$$
\begin{aligned}
t_{\text{Rev}} &= 1 + 0.3\,t_{\text{Rev}} + 0.6\,t_{\text{Test}} + 0.1\cdot 0\\
t_{\text{Test}} &= 1 + 0.2\,t_{\text{Rev}} + 0.3\,t_{\text{Test}} + 0.5\cdot 0
\end{aligned}
$$

In [4]:
t_Rev, t_Test = sp.symbols("t_Rev t_Test")

eqs_t = [sp.Eq(t_Rev,  1 + R(3, 10)*t_Rev + R(6, 10)*t_Test + R(1, 10)*0),
         sp.Eq(t_Test, 1 + R(2, 10)*t_Rev + R(3, 10)*t_Test + R(5, 10)*0)]

sol_t = sp.solve(eqs_t, [t_Rev, t_Test])
print("t_Rev  =", sol_t[t_Rev],  "≈", round(float(sol_t[t_Rev]), 2), "steps")
print("t_Test =", sol_t[t_Test], "≈", round(float(sol_t[t_Test]), 2), "steps")

t_Rev  = 130/37 ≈ 3.51 steps
t_Test = 90/37 ≈ 2.43 steps


## 3. Exercise 3 d): simulation

Each path starts in Review (index 0) and takes one step at a time, drawing the next state from
the row of the current state, until it reaches Rel (index 2) or Rej (index 3). For every path we
keep the number of steps and whether it ended in Rel.

Standard errors (Session 6): for a fraction $\hat p$ from $R$ paths, $\sqrt{\hat p(1-\hat p)/R}$;
for a mean, $s/\sqrt{R}$.

In [5]:
P3 = np.array([[0.3, 0.6, 0.0, 0.1],
               [0.2, 0.3, 0.4, 0.1],
               [0.0, 0.0, 1.0, 0.0],
               [0.0, 0.0, 0.0, 1.0]])

rng = np.random.default_rng(310)
runs = 10_000
steps = np.zeros(runs, int)
released = np.zeros(runs, bool)

for k in range(runs):
    x, n = 0, 0                          # start in Rev, no steps yet
    while x < 2:                         # 0, 1 transient; 2, 3 absorbing
        x = rng.choice(4, p=P3[x])
        n += 1
    steps[k], released[k] = n, (x == 2)

p_hat = released.mean()
se_p = np.sqrt(p_hat * (1 - p_hat) / runs)
m = steps.mean()
se_m = steps.std(ddof=1) / np.sqrt(runs)

print(f"P(release): simulated {p_hat:.4f}, SE {se_p:.4f}, exact {24/37:.4f}, "
      f"{abs(p_hat - 24/37)/se_p:.1f} SE away")
print(f"mean steps: simulated {m:.3f}, SE {se_m:.3f}, exact {130/37:.3f}, "
      f"{abs(m - 130/37)/se_m:.1f} SE away")

P(release): simulated 0.6541, SE 0.0048, exact 0.6486, 1.1 SE away
mean steps: simulated 3.502, SE 0.023, exact 3.514, 0.5 SE away


## 4. Exercise 5 c): the stationary distribution in one go

Order (1, 2, 3), $P$ read from the diagram. We need $\pi = \pi P$ with $\sum_j \pi_j = 1$.
The method from the Session 9 notebook (Section 9), in one line:

1. transpose P (the columns become the balance equations) and subtract the identity,
2. drop the last equation,
3. add a column of zeros (the right-hand side),
4. add a row of ones (the sum is 1), then `rref`.

The last column of the reduced matrix is $\pi$.

In [6]:
P5 = sp.Matrix([[R(1, 2), R(1, 4), R(1, 4)],
                [R(1, 3), 0,       R(2, 3)],
                [R(1, 2), R(1, 2), 0      ]])
r = P5.shape[0]

X = (P5.T - sp.eye(r))[0:-1, :].row_join(sp.zeros(r - 1, 1)).col_join(sp.ones(1, r + 1))
pi = X.rref()[0][:, -1].T                    # last column, as a row vector
print("pi =", pi, "≈", [round(float(v), 3) for v in pi])

pi = Matrix([[16/35, 9/35, 2/7]]) ≈ [0.457, 0.257, 0.286]


Check: $\pi P = \pi$, including the equation we dropped.

In [7]:
print("pi P =", pi * P5)
print("pi P == pi:", pi * P5 == pi)

pi P = Matrix([[16/35, 9/35, 2/7]])
pi P == pi: True


## 5. Exercise 5 d): is $\pi$ the limiting distribution?

The chain is finite, irreducible and aperiodic, so every row of $P^{n}$ tends to $\pi$.
Here we watch it happen, with NumPy (`matrix_power`, not `P**n`).

In [8]:
P5_np = np.array(P5, dtype=float)
pi_np = np.array(pi, dtype=float).ravel()

for n in [1, 5, 10, 20]:
    Pn = np.linalg.matrix_power(P5_np, n)
    print(f"n = {n}:\n{Pn}\n  largest distance to pi: {np.abs(Pn - pi_np).max():.4f}\n")

n = 1:
[[0.5    0.25   0.25  ]
 [0.3333 0.     0.6667]
 [0.5    0.5    0.    ]]
  largest distance to pi: 0.3810

n = 5:
[[0.4575 0.2582 0.2843]
 [0.4485 0.2274 0.3241]
 [0.4644 0.2821 0.2535]]
  largest distance to pi: 0.0384

n = 10:
[[0.4571 0.2571 0.2858]
 [0.4577 0.259  0.2834]
 [0.4567 0.2556 0.2877]]
  largest distance to pi: 0.0024

n = 20:
[[0.4571 0.2571 0.2857]
 [0.4571 0.2571 0.2857]
 [0.4571 0.2571 0.2857]]
  largest distance to pi: 0.0000



## 6. Exercise 6: the reliability chain in the long run

The Working/Failed chain from Session 8, Exercise 3, order (W, F). The same one-line method
as in Section 4, then the mean return time $r_F = 1/\pi_F$.

In [9]:
P6 = sp.Matrix([[R(98, 100), R(2, 100)],
                [R(30, 100), R(70, 100)]])
r = P6.shape[0]

X = (P6.T - sp.eye(r))[0:-1, :].row_join(sp.zeros(r - 1, 1)).col_join(sp.ones(1, r + 1))
pi6 = X.rref()[0][:, -1].T
print("pi = (pi_W, pi_F) =", pi6, "≈", [float(v) for v in pi6])
print("r_F = 1/pi_F =", 1 / pi6[1], "hours")

pi = (pi_W, pi_F) = Matrix([[15/16, 1/16]]) ≈ [0.9375, 0.0625]
r_F = 1/pi_F = 16 hours
